# Türkçe Neden Zor?

- Eklemeli (agglutinative) - tek kelime, çok ek:
    - ev + ler + imiz + den -> tek kelime: evlerimizden

- Büyük harf kuralları İngilizce'den farklı (i/ İ, ı , I)

- spaCy'nin Resmi Türkçe modeli YOK

# Araç Haritası

İş                        Türkçe Araç                 Not
- Lemmatization              Zeyrek             Türkçe morfol(Zemberek portu)
- NER                          Stanza            Standford, resmi Türkçe desteği
- Sınıflandırma            TF-IDF + LogReg        İngilizce'deki aynı pipeline, Türkçe stopwords

In [4]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger('zeyrek.rulebasedanaalyzer').setLevel(logging.ERROR)
import zeyrek

analiz = zeyrek.MorphAnalyzer()

In [8]:
def kok(kelime):
    # lemmatize() -> [(kelime, [olasi kokler])]; ilk analizin ilk kokunu al
    r = analiz.lemmatize(kelime)
    return r[0][1][0].lower() if r and r[0][1] else kelime

print("Kelime -> kök (lemma):")
for k in ["kitaplarımızdan","evlerinde","koşuyorlar","güzeldi","arabası"]:
    print(f"    {k:<18} ->  {kok(k)}")

# WordNet (İngilizce) bunları yapamazdı; Zeyerk köke iner.

APPENDING RESULT: <(Kitap_Noun_Prop)(-)(kitap:nounProper_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(kitap_Noun)(-)(kitap:noun_S + lar:a3pl_S + ımız:p1pl_S + dan:abl_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + a3sg_S + leri:p3pl_S + nde:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + in:p2sg_S + de:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + i:p3sg_S + nde:loc_ST)>
APPENDING RESULT: <(ev_Noun)(-)(ev:noun_S + ler:a3pl_S + i:p3pl_S + nde:loc_ST)>
APPENDING RESULT: <(koşmak_Verb)(-)(koş:verbRoot_S + uyor:vProgYor_S + lar:vA3pl_ST)>
APPENDING RESULT: <(güzel_Adj)(-)(güzel:adjectiveRoot_ST + adjZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>
APPENDING RESULT: <(güzel_Noun)(-)(güzel:noun_S + a3sg_S + pnon_S + nom_ST + nounZeroDeriv_S + nVerb_S + di:nPast_S + nA3sg_ST)>
APPENDING RESULT: <(araba_Noun)(-)(araba:noun_S + a3sg_S + sı:p3sg_S + nom_ST)>
APPENDING RESULT: <(araba_Adj)(-)(araba:adjectiveRoot_ST + adjZeroDeriv_S + noun_

Kelime -> kök (lemma):
    kitaplarımızdan    ->  kitap
    evlerinde          ->  ev
    koşuyorlar         ->  koşmak
    güzeldi            ->  güzel
    arabası            ->  araba


In [9]:
# 2) Türçe NEW - Stanza (resmi Türkçe desteği var; spaCy'nin Türkçe modeli Yok)
import stanza
# stanza.download('tr') # ilk seferde modeli indirir (kurulumda yaptık)
stanza.download("tr")
nlp_tr = stanza.Pipeline('tr', processors= 'tokenize,ner', verbose= False)

metin = "Baran Haydar Ankara'da Turkcell yöneticileriyle görüştü."
doc = nlp_tr(metin)

print("Varlık -> tip:")
for e in doc.ents:  # doc.ents -> bulunan varlıklar
    print(f"    {e.text:<16} -> {e.type}") # .text = metin, .type = PERSON/LOCATION/ORGANIZATION

2026-11-14 16:56:10 INFO: Downloaded file to /Users/baran/Library/Caches/stanza/1.15.0/resources/resources.json
2026-11-14 16:56:10 INFO: Downloading default packages for language: tr (Turkish) ...
2026-11-14 17:02:52 INFO: Downloaded file to /Users/baran/Library/Caches/stanza/1.15.0/resources/tr/default.zip
2026-11-14 17:02:53 INFO: Finished downloading models and saved to /Users/baran/Library/Caches/stanza/1.15.0/resources


Varlık -> tip:
    Baran Haydar     -> PERSON
    Ankara'da        -> LOCATION
    Turkcell         -> ORGANIZATION


In [10]:
# 3) TÜRKÇE DUYGU ANALİZİ - TF-IDF + Logistic Regression (aynı pipeline, Türkçe metin)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import nltk
nltk.download('stopwords', quiet = True) # NLTK Türkçe stopword listesi (durak kelimeler)
from nltk.corpus import stopwords
tr_stop = stopwords.words('turkish')

# Küçük DEMO veri seti (gerçek projede binlerce yorum olur)

pozitif = [
    "ürün harika, çok memnun kaldım", "kargo çok hızlı geldi teşekkürler",
    "kalitesi muhteşem kesinlikle tavsiye ederim", "fiyatına göre çok başarılı",
    "beklediğimden çok daha iyi çıktı", "mükemmel bir alışveriş deneyimiydi",
    "paketleme özenliydi ürün sağlam geldi", "tam aradığım ürün çok beğendim",
    "performansı süper hiç sorun yaşamadım","ailecek bayıldık herkese öneririm",
    "rengi ve kalitesi fotoğraftaki gibi memnunum","uzun süredir kullanıyorum gayet iyi",
    "destek ekibi çok ilgili sorunu hemen çözdüler","kullanımı çok kolay pratik",
    "bu fiyata bu kalite inanılmaz", "kesinlikle paranızın karşılığını veriyor",
    "hızlı teslimat ve sağlam paket bravo", "çok şık duruyor çok memnunum",
    "kaliteli malzeme uzun ömürlü görünüyor","tekrar alırım gönül rahatlıgıyla"
]
negatif = [
    "berbat bir deneyimdi kesinlikle almayın","ürün kırık geldi çok kötü",
    "kargo bir haftaya geç geldi rezalet","kalitesi çok düşük paranıza yazık",
    "fotoğraftakiyle alakası yok hayal kırıklığı","bir hafta içinde bozuldu pişmanım",
    "destek ekibi ilgisiz kimse dönmedi","fiyatına hiç değmez almayın",
    "paketleme kötüydü ürün hatalıydı","çok kalitesiz iade ediyorum",
    "yanlış ürün gönderdiler hala çözülmedi","kesinlikle tavsiye etmiyorum",
    "ilk kullanımda arızalandı çok sinir bozucu","boşa harcanmış para resmen",
    "malzeme ucuz hissettiriyor memnun değilim", "geç geldi üstüne bir de bozuk",
    "rengi soldu kısa sürede berbat kalite","asla tekrar almam çok kötüydü"
]

yorumlar = pozitif + negatif
etiketler = [1] * len(pozitif) + [0]*len(negatif)

# Tüm veriyle eğit (küçük demo seti - train/test ölçümü anlamlı olmaz, ml3'teki dersi hatırla)
pipe = Pipeline([("tfidf", TfidfVectorizer(stop_words= tr_stop)),
                 ("clf", LogisticRegression(max_iter= 1000))])

pipe.fit(yorumlar, etiketler)

# Modelin HİÇ görmediği yeni cümleler -> doğru ayırıyor mu?
print("Yeni cümle tahminleri:")
for c in ["ürün gerçekten çok kaliteli bayıldım",
          "tam bir hayal kırıklığı asla almayın",
          "fiyatı uygun kargo hızlı çok memnunum"]:
    p = pipe.predict([c])[0]
    print(f"    {"POZITIF" if p == 1 else 'NEGATIF':<7} <- '{c}'")


# Modelin öğrendiği en ayırt edici kelimeler (yorumlanabilirlik)
import numpy as np
kelimeler = pipe. named_steps["tfidf"].get_feature_names_out()
coef = pipe.named_steps["clf"].coef_[0]
print("\nEn POZİTİF kelimeler:", [kelimeler[i] for i in np.argsort(coef)[::-1][:6]])
print("En NEGATİF kelimeler:", [kelimeler[i] for i in np.argsort(coef[:6])])

# Not: Bu küçük demo. Gerçek projede binlerce yorum + train/tesr split ile ölçülür
# (DS-38 İngilizce duygu projesindeki gibi). Buradaki amaç: AYNI pipeline Türkçe'de de çalışıyor.

Yeni cümle tahminleri:
    POZITIF <- 'ürün gerçekten çok kaliteli bayıldım'
    NEGATIF <- 'tam bir hayal kırıklığı asla almayın'
    POZITIF <- 'fiyatı uygun kargo hızlı çok memnunum'

En POZİTİF kelimeler: ['memnunum', 'hızlı', 'sağlam', 'iyi', 'uzun', 'başarılı']
En NEGATİF kelimeler: ['almayın', 'almam', 'alakası', 'ailecek', 'alırım', 'alışveriş']


# ÖZET 

- Türkçe eklemeli dil -> dile özel araç şart, spaCy Türkçe modeli YOK

- Lemmatization -> Zeyrek (kitaplarımızdan -> kitap)

- NER -> Stanza (resmi Türkçe: PERSON/LOCATION/ORGANIZATION)

- Sınıflandırma -> TF-IDF + LogReg(aynı pipeline, Türkçe stopword)

- İleri -> BERTurk / FastText(büyük, daha doğru)